# Day 05: CUDA Kernels, Kernel Selection & Kernel Fusion
> *Inference Engineering* — Chapter 4.1 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 04 (Arithmetic Intensity)


## Concept Overview

A CUDA kernel is a function that runs in parallel across thousands of GPU threads. PyTorch dispatches operations to kernels via its dispatcher based on device, dtype, and layout. Kernel fusion combines multiple operations into a single kernel pass, reducing memory round-trips. For memory-bound operations (like elementwise activations), fusion is transformative: instead of reading and writing tensors N times, you read once, compute all ops in registers, and write once. FlashAttention is the canonical example of kernel fusion in LLM inference.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'PyTorch version: {torch.__version__}')


## 1. Kernel Dispatch Overhead

Each PyTorch operation incurs kernel launch overhead (~5-20 µs). For small tensors, this overhead dominates. Measuring dispatch cost reveals where fusion matters most.


In [ ]:
def benchmark_op(fn, *args, warmup=10, iters=100, device='cpu'):
    for _ in range(warmup):
        fn(*args)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(*args)
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e6  # microseconds

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sizes = [64, 256, 1024, 4096, 16384]
results = {'unfused': [], 'fused_approx': []}

for n in sizes:
    x = torch.randn(n, device=device)
    # Unfused: separate ops, each hitting HBM
    t_unfused = benchmark_op(lambda a: F.gelu(a * 2.0 + 1.0), x, device=device)
    # Simulate fused (torch.compile would actually fuse this)
    results['unfused'].append(t_unfused)

print(f'{"Size":>8} {"Unfused (µs)":>15}')
for n, t in zip(sizes, results['unfused']):
    print(f'{n:>8} {t:>15.1f}')
print(f'\nKernel launch overhead is ~constant for small tensors')


## 2. Kernel Fusion via torch.compile

PyTorch's `torch.compile` uses TorchInductor to fuse pointwise operations into a single kernel. This eliminates intermediate memory writes and reduces kernel launch count.


In [ ]:
def gelu_scale_bias(x, scale, bias):
    """Three elementwise ops: scale, add bias, gelu."""
    return F.gelu(x * scale + bias)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(1024 * 1024, device=device, dtype=torch.float16)
scale = torch.tensor(2.0, device=device, dtype=torch.float16)
bias = torch.tensor(0.5, device=device, dtype=torch.float16)

# Eager (unfused)
t_eager = benchmark_op(gelu_scale_bias, x, scale, bias, device=device)
print(f'Eager (unfused): {t_eager:.1f} µs')

if device == 'cuda':
    try:
        compiled = torch.compile(gelu_scale_bias, mode='reduce-overhead')
        # Warmup compile
        for _ in range(5):
            compiled(x, scale, bias)
        torch.cuda.synchronize()
        t_compiled = benchmark_op(compiled, x, scale, bias, device=device)
        print(f'Compiled (fused): {t_compiled:.1f} µs')
        print(f'Speedup from fusion: {t_eager/t_compiled:.2f}x')
    except Exception as e:
        print(f'torch.compile: {e}')
        print('Fusion benefit: typically 1.5-3x for pointwise ops')
else:
    print('Run on GPU to see fusion benefits')


## 3. SwiGLU Fusion — A Real LLM Example

SwiGLU is the FFN activation used in Llama, Mistral, and most modern LLMs. The unfused version requires 3 kernel launches; fused version is 1.

$$\text{SwiGLU}(x) = (xW_1) \cdot \sigma(xW_3) \cdot W_2$$


In [ ]:
class SwiGLU_Unfused(torch.nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = torch.nn.Linear(d_model, d_ff, bias=False)
        self.w3 = torch.nn.Linear(d_model, d_ff, bias=False)
        self.w2 = torch.nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.w1(x))   # kernel 1: silu
        up   = self.w3(x)            # kernel 2: matmul
        return self.w2(gate * up)    # kernel 3: elementwise mul + matmul

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SwiGLU_Unfused(d_model=4096, d_ff=14336).to(device).half()
x = torch.randn(1, 32, 4096, device=device, dtype=torch.float16)  # batch=1, seq=32

t_unfused = benchmark_op(model, x, device=device)
print(f'SwiGLU unfused: {t_unfused:.1f} µs (batch=1, seq=32)')

if device == 'cuda':
    try:
        compiled_model = torch.compile(model, mode='reduce-overhead')
        for _ in range(3): compiled_model(x)
        torch.cuda.synchronize()
        t_fused = benchmark_op(compiled_model, x, device=device)
        print(f'SwiGLU compiled: {t_fused:.1f} µs')
        print(f'Speedup: {t_unfused/t_fused:.2f}x')
    except Exception as e:
        print(f'Compile note: {e}')

# Count theoretical kernel launches
print('\nKernel launch analysis:')
print('  Unfused SwiGLU: ~5 kernel launches (2x matmul, silu, mul, matmul)')
print('  Fused SwiGLU:   ~3 kernel launches (2x matmul, fused silu+mul)')
print('  Benefit: 2 fewer global memory round-trips')


## 4. FlashAttention as Kernel Fusion

Standard attention requires 4 HBM round-trips (QK^T, softmax, AV, output). FlashAttention fuses all into 1 pass using on-chip SRAM tiling — the key insight for memory-bound inference.


In [ ]:
# Compare memory access patterns: standard vs flash attention
def attention_memory_accesses(seq_len, d_head, batch, num_heads, dtype_bytes=2):
    N, d = seq_len, d_head
    H, B = num_heads, batch

    # Standard attention HBM accesses
    qkv_read = 3 * B * H * N * d * dtype_bytes
    attn_matrix_write = B * H * N * N * dtype_bytes   # write QK^T
    attn_matrix_read  = B * H * N * N * dtype_bytes   # read for AV
    output_write = B * H * N * d * dtype_bytes
    standard_total = qkv_read + attn_matrix_write + attn_matrix_read + output_write

    # FlashAttention HBM accesses (tiled, no full attn matrix)
    flash_total = qkv_read + output_write  # only read QKV, write output

    return standard_total / 1e9, flash_total / 1e9  # GB

print(f'{"Seq Len":>8} {"Standard (GB)":>15} {"Flash (GB)":>12} {"Reduction":>12}')
print('-' * 50)
for seq in [512, 1024, 2048, 4096, 8192, 16384]:
    std, flash = attention_memory_accesses(seq, d_head=128, batch=1, num_heads=32)
    print(f'{seq:>8} {std:>15.3f} {flash:>12.3f} {std/flash:>11.1f}x')


## Experiments: Try These

1. **Profile with torch.profiler**: Wrap a forward pass in `torch.profiler.profile` and count kernel launches before vs after `torch.compile`.
2. **Custom Triton kernel**: Write a fused `x * sigmoid(x)` (SiLU) Triton kernel and compare bandwidth to PyTorch eager.
3. **Fusion sensitivity**: Measure kernel fusion benefit as a function of tensor size. At what size does the fusion benefit disappear (when memory bandwidth saturates)?


## Key Takeaways

- CUDA kernels are GPU functions; each PyTorch op dispatches at least one, incurring launch overhead and HBM round-trips.
- Kernel fusion combines multiple pointwise ops into a single kernel, reducing global memory traffic — critical for memory-bound operations.
- `torch.compile` with TorchInductor automatically fuses pointwise operations; the speedup is proportional to memory bandwidth saved.
- FlashAttention is the most impactful kernel fusion in LLM inference: it eliminates the $O(N^2)$ attention matrix from HBM, enabling long-context inference.

## References
- *Inference Engineering* Ch 4.1 — Philip Kiely, Baseten Books 2026
- Dao et al. (2022), "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"
- PyTorch TorchInductor documentation
